# RAG Query Examples

This notebook demonstrates how to use the RAG pipeline to answer questions about DSA projects.

In [ ]:
import sys
sys.path.append('..')

from src.config import config
from src.rag.vectorstore import VectorStore
from src.rag.embeddings import EmbeddingManager
from src.rag.pipeline import RAGPipeline
from loguru import logger
import json

## 1. Initialize RAG Pipeline

In [ ]:
# Validate configuration
if not config.validate():
    raise ValueError("Configuration validation failed. Please check your .env file.")

print("✅ Configuration validated")

In [ ]:
# Initialize components
print("Initializing RAG pipeline components...")

vector_store = VectorStore(
    persist_directory=config.VECTOR_DB_PATH,
    collection_name='confluence_docs'
)

embedding_manager = EmbeddingManager(model_name=config.EMBEDDING_MODEL)

pipeline = RAGPipeline(
    vector_store=vector_store,
    embedding_manager=embedding_manager,
    iliad_api_key=config.ILIAD_API_KEY,
    iliad_api_url=config.ILIAD_API_URL,
    top_k=config.TOP_K_RESULTS
)

print(f"✅ RAG pipeline initialized")
print(f"Vector store contains {vector_store.count()} documents")

## 2. Single Query Example

In [ ]:
# Ask a question
question = "What is the purpose of the customer segmentation project?"

print(f"Question: {question}\n")
print("Querying RAG pipeline...\n")

result = pipeline.query(question)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("SOURCES:")
print("=" * 80)
for i, source in enumerate(result['sources'], 1):
    print(f"{i}. {source['title']}")
    print(f"   Type: {source['type']}")
    print(f"   URL: {source['url']}\n")

## 3. Multiple Query Examples

In [ ]:
# Define multiple questions
questions = [
    "What data science projects are documented in Confluence?",
    "Which projects use machine learning models?",
    "What GitHub repositories are available?",
    "What technologies and tools are commonly used?",
]

print("Running batch queries...\n")

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"QUESTION {i}: {question}")
    print(f"{'='*80}")
    
    result = pipeline.query(question, n_results=3)
    
    print("\nANSWER:")
    print(result['answer'])
    
    print(f"\nSOURCES: {len(result['sources'])}")
    for source in result['sources']:
        print(f"  - {source['title']} ({source['type']})")

## 4. Retrieve and Inspect Source Documents

In [ ]:
# Use a query to retrieve documents
query = "machine learning projects"

print(f"Retrieving documents for: {query}\n")

retrieved = pipeline.retrieve_relevant_documents(query, n_results=5)

print(f"Retrieved {len(retrieved['documents'])} documents:\n")

for i, (doc, meta, dist) in enumerate(zip(
    retrieved['documents'],
    retrieved['metadatas'],
    retrieved['distances']
), 1):
    print(f"{i}. {meta['title']}")
    print(f"   Source: {meta['source_type']}")
    print(f"   Distance: {dist:.4f}")
    print(f"   URL: {meta['url']}")
    print(f"   Preview: {doc[:150]}...\n")

## 5. Custom Query with Different Parameters

In [ ]:
# Query with more context documents
question = "Tell me about the data sources used in projects"

print(f"Question: {question}\n")
print("Using 10 context documents for more comprehensive answer...\n")

result = pipeline.query(question, n_results=10)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(result['answer'])
print(f"\n(Answer generated from {len(result['retrieved_documents'])} source documents)")

## 6. Export Results

In [ ]:
# Save a query result to JSON
output_file = "query_result.json"

question = "What are the key objectives of DSA projects?"
result = pipeline.query(question)

# Prepare data for export (remove non-serializable fields)
export_data = {
    'question': result['question'],
    'answer': result['answer'],
    'sources': result['sources'],
    'num_documents': len(result['retrieved_documents']),
    'distances': [float(d) for d in result['distances']]
}

with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Query result exported to {output_file}")

## 7. Interactive Query Interface

In [ ]:
# Simple interactive loop
def interactive_query():
    """Interactive query interface for the notebook."""
    print("Interactive RAG Query Interface")
    print("Type 'quit' or 'exit' to stop\n")
    
    while True:
        question = input("\nYour question: ").strip()
        
        if question.lower() in ['quit', 'exit', '']:
            print("Goodbye!")
            break
        
        try:
            result = pipeline.query(question)
            print("\nAnswer:")
            print("-" * 80)
            print(result['answer'])
            print("\nSources:")
            for source in result['sources']:
                print(f"  - {source['title']}: {source['url']}")
        except Exception as e:
            print(f"Error: {str(e)}")

# Uncomment to run interactive mode
# interactive_query()

## Summary

This notebook demonstrated:
- ✅ Initializing the RAG pipeline
- ✅ Asking single questions
- ✅ Running batch queries
- ✅ Retrieving and inspecting source documents
- ✅ Customizing query parameters
- ✅ Exporting results

You can now use the RAG pipeline to answer questions about your DSA projects!